# vgscp — full REAL multi-seed run (E1–E4) on Colab GPU

Runs the four experiments end-to-end on **real** Waterbirds + CUB-200 data with frozen CLIP ViT-B/32 features (encoded once, cached) and logistic heads — **no large-model training**. Run top-to-bottom on a **GPU** runtime (`Runtime → Change runtime type → GPU`).

Honesty: the pre-committed verdicts (`eval/e1_verdict.py`, `eval/scacp_gate.py`) are LOCKED. Whatever the real multi-seed numbers are — including FALLBACK / ties / softenings — is what gets reported.

## 0. Parameters — **EDIT THESE**

In [ ]:
# ===================== EDIT THESE =====================
REPO_SOURCE    = "git"          # "git" or "drive"
REPO_URL       = "https://github.com/<YOUR_USER>/vgscp.git"   # EDIT (private: https://<TOKEN>@github.com/<user>/vgscp.git)
REPO_BRANCH    = "main"
REPO_DRIVE_ZIP = "/content/drive/MyDrive/vgscp.zip"           # used only if REPO_SOURCE=="drive"

DRIVE_CACHE    = "/content/drive/MyDrive/vgscp_cache"  # datasets + CLIP feature cache persisted here
SEEDS          = 10                                    # >=10 random cal/test splits per spec

# Dataset URLs — EDIT IF URL CHANGES
WATERBIRDS_URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CUB_URL        = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"
# ======================================================
import os, time, subprocess, sys
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)
def run_module(mod_args, label):
    t = time.time(); print(f"\n===== {label} =====")
    p = subprocess.run([sys.executable, "-m", *mod_args])
    dt = time.time() - t; print(f"[{label}] exit={p.returncode}  wall={dt/60:.1f} min")
    if dt > 4.5 * 3600: print(f"[WARN] {label} approaching the 5h cap")
    return p.returncode

## 1. GPU check + install

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[WARN] No GPU — CLIP encode will be slow. Set Runtime->GPU.")
import subprocess
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn pandas matplotlib", shell=True)

## 2. Mount Drive (persist datasets + CLIP feature cache across restarts)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CACHE, exist_ok=True)
print("cache dir:", DRIVE_CACHE)

## 3. Get the repo

In [ ]:
REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
    subs = [d for d in os.listdir(REPO_DIR) if os.path.isdir(f"{REPO_DIR}/{d}")]
    if len(subs) == 1 and not os.path.exists(f"{REPO_DIR}/scripts"):
        inner = f"{REPO_DIR}/{subs[0]}"; sh(f"shopt -s dotglob && mv {inner}/* {REPO_DIR}/ && rmdir {inner}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print("repo:", os.getcwd())

## 4. Datasets — download (cached to Drive) + extract, set env vars

In [ ]:
def fetch(url, drive_name, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    tarball = os.path.join(DRIVE_CACHE, drive_name)
    if not os.path.exists(tarball):
        sh(f"wget -q -O '{tarball}' '{url}'")
    else:
        print("cached tarball:", tarball)
    sh(f"tar -xzf '{tarball}' -C '{extract_to}'")
    return extract_to

fetch(WATERBIRDS_URL, "waterbirds.tar.gz", "/content/data/waterbirds")
fetch(CUB_URL, "CUB_200_2011.tgz", "/content/data/cub")
os.environ["WATERBIRDS_ROOT"] = "/content/data/waterbirds"
os.environ["CUB_ROOT"] = "/content/data/cub"
# point the repo CLIP feature cache at Drive so re-runs skip re-encoding
sh("rm -rf results/cache_clip"); os.makedirs("results", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/clip", exist_ok=True)
sh(f"ln -s {DRIVE_CACHE}/clip results/cache_clip")
print("WATERBIRDS_ROOT=", os.environ["WATERBIRDS_ROOT"]); print("CUB_ROOT=", os.environ["CUB_ROOT"])

## 5. Encode + cache CLIP features ONCE (later cells reuse the cache)

In [ ]:
t = time.time()
from config_util import load_config
from experiments.real_data import load_real_bundle
cfg = load_config("configs/cub200_frontier.yaml")
bundle = load_real_bundle(cfg, seed=0)
print("feature shapes:", {k: v.shape for k, v in bundle.features.items()})
print("species present:", bundle.info["n_species_present"], "| CUB join:", bundle.info["cub_join"]["coverage"])
print(f"[encode+cache] wall={(time.time()-t)/60:.1f} min (cached to Drive)")

## 6. CORRECTED unified 2×2 — replaces E1 + E3 (run spec v2)
Complete `{feature, concept} × {split, Mondrian}` grid, **one score function compared across
representations** (APS primary; RAPS/THR appendix), clean-CUB feature head (§2a, ≥0.55 sanity gate),
**image-derived predicted concepts** (§2b: `cbm` primary, `zeroshot` appendix), ρ_cal=0.95, finer
ρ_test sweep, ≥10 seeds. Produces both views (gap-vs-ρ; worst-cov-vs-set-size frontier) and the
pre-committed **group-free-substitution** verdict `R`. **Report whatever it produces — the
kill-switch (FALLBACK) is LIVE.** If the §2a clean-CUB top-1 prints below ~0.5, STOP and report the
diagnosis (do not proceed with a broken head).

In [ ]:
# primary: image-derived CBM concept; appendix: CLIP zero-shot concept (leakage-free robustness check)
run_module(["scripts.run_unified_2x2", "--config", "configs/cub200_frontier.yaml",
            "--seeds", str(SEEDS), "--concept-source", "cbm"], "UNIFIED-cbm")
run_module(["scripts.run_unified_2x2", "--config", "configs/cub200_frontier.yaml",
            "--seeds", str(SEEDS), "--concept-source", "zeroshot", "--out", "results/unified_zeroshot"],
           "UNIFIED-zeroshot")

import pandas as pd, json
from IPython.display import Image, display
r = json.load(open("results/unified/unified_results.json"))
print(f"§2a clean-CUB feature top-1 = {r['acc_control'].get('matched_feat_top1')}  |  "
      f"heads: feature={r['feat_top1']:.3f}  concept[{r['concept_source']}]={r['cpt_top1']:.3f}")
print("VERDICT (APS):", r["verdicts"]["APS"]["label"]); print(r["verdicts"]["APS"]["rationale"])
df = pd.read_csv("results/unified/unified_2x2.csv")
g = (df[df.score == "APS"].groupby(["representation", "scheme", "test_corr"])
     .agg(worst_cov=("worst_cov", "mean"), set_size=("mean_set_size", "mean"),
          gap=("cov_gap", "mean"), marg=("marg_cov", "mean")).round(3))
display(g)
for nm in ("u2x2_gap_vs_rho_APS", "u2x2_frontier_APS"):
    p = f"results/figures/{nm}.png"
    if os.path.exists(p): display(Image(p))

## 7. §2a head diagnosis + §2e accuracy-controlled efficiency
The former **E3** (correlation-strength shift, worst-group coverage + gap) is **folded into the
unified 2×2 above** (the gap-vs-ρ figure *is* the E3 view, now with the same score across
representations and the complete grid). Below: the §2a clean-CUB feature-head sanity number and the
§2e accuracy-matched efficiency (concept vs feature set size) — the efficiency claim is reported
**only with the accuracy caveat attached**.

In [ ]:
import json
r = json.load(open("results/unified/unified_results.json"))
ac, eff = r["acc_control"], r["efficiency"]
print(f"§2a clean-CUB feature linear-probe top-1 must clear ~0.55 (prior broken run: 0.182).")
print(f"   heads: feature={r['feat_top1']:.3f}  concept[{r['concept_source']}]={r['cpt_top1']:.3f}")
print(f"§2e accuracy-match feasible={ac['feasible']}  (n_matched_classes={ac['n_matched_classes']}, "
      f"tol={ac['tol']})")
if not ac["feasible"]:
    print("   -> efficiency result is CONFOUNDED BY ACCURACY (labelled as such; no representation-"
          "driven efficiency claim without the control).")
import pandas as pd
display(pd.DataFrame(eff["per_rho"]).round(3))
# mechanism main effect (reported, not suppressed): Mondrian's gap reduction per representation
v = r["verdicts"]["APS"]
print(f"Mechanism main effect (Mondrian gap reduction, sweep mean): feature={v['sweep_mean_mech_feat']:+.3f}"
      f"  concept={v['sweep_mean_mech_cpt']:+.3f}  |  recovered fraction R sweep-mean={v['sweep_mean_R']:.2f}")

## 8. E2 — verifiability collapse  *(clean in the prior run — NOT re-run per spec v2 §3)*
E2 was clean; spec v2 says **reporting fixes only, no re-run**. The corrected reporting is in
`RESULTS_v2.md §5` (clean-space **V_full 0.969 > concept_trust 0.957**; collapse is **mixed + Morgana-
ON**; V_comp Morgana-OFF ≈ 0.77 both spaces; contamination AUROC concept_trust ≈ 0.688). The cell
below is retained **only to regenerate** the raw metrics if needed — it does not change the verdict.

In [ ]:
run_module(["scripts.run_e2_verifiability_multiseed", "--config", "configs/premise2_waterbirds.yaml", "--seeds", str(SEEDS)], "E2")
import pandas as pd
d = pd.read_csv("results/e2/e2_verifiability_metrics.csv")
display(d.groupby(["space", "signal"])
        .agg(min_auroc=("minority_auroc", "mean"), contam=("contamination_auroc", "mean")).round(3))

## 9. E4 — scacp 312-attribute locked gate  *(clean negative — NOT re-run per spec v2 §3)*
E4 was a clean negative; spec v2 says **reporting fixes only**. Corrected reporting (`RESULTS_v2.md
§5`): **1/311 pass** (not 0/312), **median differential-noise AUROC 0.596** (not ≈0.48) — a clean
negative; the single pass is expected at this multiplicity. The cell below only regenerates the scan.

In [ ]:
run_module(["scripts.run_e4_scacp_gate", "--real", "--config", "configs/cub200_frontier.yaml"], "E4")
import json, pandas as pd
r = json.load(open("results/e4/e4_results.json"))
print(f"GATE PASSES: {r['n_pass']}/{r['n_attributes']}  median diff-noise AUROC={r['median_diff_auroc']:.3f}")
print(f"per-criterion: diff>=0.70:{r['pass_diff']}  gap>=0.03:{r['pass_gap']}  support>=100:{r['pass_support']}")
d = pd.read_csv("results/e4/e4_scacp_gate_scan.csv")
display(d["diff_noise_auroc"].describe().round(3))

## 10. Consolidate + copy to Drive + zip for download

In [ ]:
import json, datetime, platform
summary = {"date": str(datetime.date.today()), "seeds": SEEDS, "platform": platform.platform()}
# corrected unified 2×2 (replaces E1+E3); E2/E4 reporting-only
for tag, p in (("unified_cbm", "results/unified/unified_results.json"),
               ("unified_zeroshot", "results/unified_zeroshot/unified_results.json"),
               ("e2", "results/e2/e2_results.json"), ("e4", "results/e4/e4_results.json")):
    if os.path.exists(p): summary[tag] = json.load(open(p))
open("results/REAL_RUN_v2_SUMMARY.json", "w").write(json.dumps(summary, indent=2, default=str))
out = f"{DRIVE_CACHE}/results_real_v2"
sh(f"rm -rf {out} && cp -r results {out}")
for f in ("RESULTS_v2.md", "BLOCKERS_v2.md", "CHANGELOG_v2.md", "RESULTS.md", "BLOCKERS.md"):
    if os.path.exists(f): sh(f"cp {f} {out}/ 2>/dev/null || true")
sh(f"cd {DRIVE_CACHE} && zip -qr results_real_v2.zip results_real_v2")
print("Saved to Drive:", out, "and", f"{DRIVE_CACHE}/results_real_v2.zip")
try:
    from google.colab import files; files.download(f"{DRIVE_CACHE}/results_real_v2.zip")
except Exception as e: print("download skipped:", e)
print("\nNOTE: report whatever the unified 2×2 produces — GREEN or FALLBACK. If the §2a clean-CUB "
      "top-1 < ~0.5, STOP and report the diagnosis per BLOCKERS_v2.md (do not proceed with a broken head).")